# EDA PARA PREDICCIÓN DE VENTAS CON REDES NEURONALES RECURRENTES
# FUENTA : https://www.kaggle.com/competitions/m5-forecasting-accuracy/data

# IMPORTAMOS LIBRERIAS

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
PATH_CALENDAR = '/content/drive/MyDrive/CODIGOG7/MODULO9/2-EDA/calendar.csv'
PATH_SALES = '/content/drive/MyDrive/CODIGOG7/MODULO9/2-EDA/sales_train_evaluation.csv.zip'

# CARGAMOS DATASET CALENDAR

In [3]:
calendar = pd.read_csv(PATH_CALENDAR)
calendar.head(5)

,date,wm_yr_wk,weekday,wday,month,year,d,event_name_1,event_type_1,event_name_2,event_type_2,snap_CA,snap_TX,snap_WI
0,2011-01-29,11101,Saturday,1,1,2011,d_1,NaN,NaN,NaN,NaN,0,0,0
1,2011-01-30,11101,Sunday,2,1,2011,d_2,NaN,NaN,NaN,NaN,0,0,0
2,2011-01-31,11101,Monday,3,1,2011,d_3,NaN,NaN,NaN,NaN,0,0,0
3,2011-02-01,11101,Tuesday,4,2,2011,d_4,NaN,NaN,NaN,NaN,1,1,0
4,2011-02-02,11101,Wednesday,5,2,2011,d_5,NaN,NaN,NaN,NaN,1,0,1


# CARGAMOS DATASET SALES

In [4]:
import zipfile

with zipfile.ZipFile(PATH_SALES, 'r') as zip_ref:
    zip_ref.extractall('./')
print('sales_train_evaluation.csv extracted ')

sales_train_evaluation.csv extracted 


In [5]:
sales = pd.read_csv("/content/sales_train_evaluation.csv")
sales

,id,item_id,dept_id,cat_id,store_id,state_id,d_1,d_2,d_3,d_4,...,d_1932,d_1933,d_1934,d_1935,d_1936,d_1937,d_1938,d_1939,d_1940,d_1941
0,HOBBIES_1_001_CA_1_evaluation,HOBBIES_1_001,HOBBIES_1,HOBBIES,CA_1,CA,0,0,0,0,...,2,4,0,0,0,0,3,3,0,1
1,HOBBIES_1_002_CA_1_evaluation,HOBBIES_1_002,HOBBIES_1,HOBBIES,CA_1,CA,0,0,0,0,...,0,1,2,1,1,0,0,0,0,0
2,HOBBIES_1_003_CA_1_evaluation,HOBBIES_1_003,HOBBIES_1,HOBBIES,CA_1,CA,0,0,0,0,...,1,0,2,0,0,0,2,3,0,1
3,HOBBIES_1_004_CA_1_evaluation,HOBBIES_1_004,HOBBIES_1,HOBBIES,CA_1,CA,0,0,0,0,...,1,1,0,4,0,1,3,0,2,6
4,HOBBIES_1_005_CA_1_evaluation,HOBBIES_1_005,HOBBIES_1,HOBBIES,CA_1,CA,0,0,0,0,...,0,0,0,2,1,0,0,2,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
30485,FOODS_3_823_WI_3_evaluation,FOODS_3_823,FOODS_3,FOODS,WI_3,WI,0,0,2,2,...,1,0,3,0,1,1,0,0,1,1
30486,FOODS_3_824_WI_3_evaluation,FOODS_3_824,FOODS_3,FOODS,WI_3,WI,0,0,0,0,...,0,0,0,0,0,0,1,0,1,0
30487,FOODS_3_825_WI_3_evaluation,FOODS_3_825,FOODS_3,FOODS,WI_3,WI,0,6,0,2,...,0,0,1,2,0,1,0,1,0,2
30488,FOODS_3_826_WI_3_evaluation,FOODS_3_826,FOODS_3,FOODS,WI_3,WI,0,0,0,0,...,1,1,1,4,6,0,1,1,1,0


# Columnas de días (d_1, d_2, ...)

In [6]:
day_cols = [c for c in sales.columns if c.startswith("d_")]

# Convertir de ancho a largo

In [7]:
sales_long = sales.melt(
    id_vars=["item_id"],
    value_vars=day_cols,
    var_name="d",
    value_name="sales"
)

# Sumar ventas de todas las tiendas para cada producto y día

In [8]:
sales_long = (
    sales_long
    .groupby(["item_id", "d"], as_index=False)
    .agg({"sales": "sum"})
)

# Unir con el calendario para obtener la fecha real

In [9]:
calendar["date"] = pd.to_datetime(calendar["date"])

dataset = sales_long.merge(
    calendar[["d", "date"]],
    on="d",
    how="left"
)

In [10]:
# Ordenar
dataset = dataset.sort_values(["item_id", "date"])

In [11]:
dataset

,item_id,d,sales,date
0,FOODS_1_001,d_1,6,2011-01-29
1053,FOODS_1_001,d_2,6,2011-01-30
1164,FOODS_1_001,d_3,4,2011-01-31
1275,FOODS_1_001,d_4,6,2011-02-01
1386,FOODS_1_001,d_5,7,2011-02-02
...,...,...,...,...
5917210,HOUSEHOLD_2_516,d_1937,1,2016-05-18
5917211,HOUSEHOLD_2_516,d_1938,1,2016-05-19
5917212,HOUSEHOLD_2_516,d_1939,3,2016-05-20
5917214,HOUSEHOLD_2_516,d_1940,4,2016-05-21


In [12]:
dataset.isnull().sum()

,0
item_id,0
d,0
sales,0
date,0


In [13]:
dataset.to_csv("m5_lstm_producto_fecha.csv", index=False)